# FORUM correlations

This notebook treats FORUM as an outcome of a ranking policy and asks where it agrees or disagrees with other summaries of the same ranking. The main plots are ranking-level plots: a point is a policy × outcome × interface/depth combination, with FORUM and nDCG averaged over stories. This avoids treating millions of comment rows as independent observations.

Questions:

1. Do top-10 and full-list FORUM agree across all rankings?
2. Does FORUM agree with nDCG across all rankings?
3. Do feature-level FORUM values from the regression rankings align with regression coefficients for the audience and editor selectors?
4. What should the equivalent ML FORUM-versus-SHAP analysis look like?

The fourth question is deliberately deferred. This notebook does **not** calculate SHAP. Existing SHAP files are inventoried only to check provenance and alignment with the current selected rankers.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'commentgap_analysis').exists() and (REPO_ROOT.parent / 'commentgap_analysis').exists():
    REPO_ROOT = REPO_ROOT.parent

PAPER1_TABLES = REPO_ROOT / 'model_output/selection_2025/paper1/reporting/tables'
FORUM_ANALYSIS_INFERENCE = REPO_ROOT / 'model_output/selection_2025/forum_ranking_analysis/inference'
FORUM_ANALYSIS_SCORES = REPO_ROOT / 'model_output/selection_2025/forum_ranking_analysis/policy_scores/policy_scores.parquet'
REGRESSION_ROOT = REPO_ROOT / 'model_output/selection_2025/regression/all'

FEATURE_LABELS = {
    'aqua_score_expected': 'AQuA aggregate score',
    'article_similarity_top3': 'Article similarity',
    'toxicity_probability': 'Toxicity',
    'log_author_prior_30d_comments': 'Participant incumbency',
    'author_prior_30d_reception_balance': 'Prior audience reception',
    'semantic_novelty_knn5': 'Comment novelty (5-NN)',
    'sentiment_positive': 'Positive sentiment',
    'sentiment_negative': 'Negative sentiment',
    'cttr': 'Lexical diversity (raw CTTR)',
    'lexdiv_length_adjusted': 'Lexical diversity (adjusted)',
    'smog_de': 'Reading difficulty (raw SMOG-DE)',
}
ORDERING_LABELS = {
    'regression_audience': 'Regression audience',
    'regression_editor': 'Regression editor',
    'xgb_metadata_audience': 'XGB metadata audience',
    'xgb_metadata_editor': 'XGB metadata editor',
    'xgb_metadata_text_audience': 'XGB metadata + text audience',
    'xgb_metadata_text_editor': 'XGB metadata + text editor',
    'neural_metadata_audience': 'Neural metadata audience',
    'neural_metadata_editor': 'Neural metadata editor',
    'neural_metadata_text_audience': 'Neural metadata + text audience',
    'neural_metadata_text_editor': 'Neural metadata + text editor',
}

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.bbox'] = 'tight'

## 1. Load canonical ranking-level products

`policy_summary.csv` supplies the canonical story-averaged FORUM estimates. The policy-score parquet is used to obtain the matching story-averaged nDCG values. We keep the primary sample and retain all ordering, reply-mode, pinning, and depth combinations.

In [ ]:
policy_summary = pd.read_csv(FORUM_ANALYSIS_INFERENCE / 'policy_summary.csv')
policy_summary = policy_summary[policy_summary['sample'].eq('primary')].copy()
policy_summary['feature_label'] = policy_summary['outcome'].map(FEATURE_LABELS).fillna(policy_summary['outcome'])

score_columns = [
    'story_id', 'policy_id', 'ordering', 'reply_mode', 'pinned', 'deployable',
    'outcome', 'outcome_role', 'depth', 'forum', 'ndcg'
]
policy_scores = pd.read_parquet(FORUM_ANALYSIS_SCORES, columns=score_columns)
# outcome_role distinguishes the AQuA secondary outcome within the score product;
# it is still included in the primary-sample policy summary and belongs in the
# all-feature comparison requested here.

ranking_metrics = (
    policy_scores
    .groupby(['policy_id', 'ordering', 'reply_mode', 'pinned', 'deployable', 'outcome', 'depth'], as_index=False)
    .agg(forum=('forum', 'mean'), ndcg=('ndcg', 'mean'), n_stories=('story_id', 'nunique'))
)
ranking_metrics['feature_label'] = ranking_metrics['outcome'].map(FEATURE_LABELS).fillna(ranking_metrics['outcome'])
ranking_metrics['ordering_label'] = ranking_metrics['ordering'].map(ORDERING_LABELS).fillna(ranking_metrics['ordering'])

# Check that the independent aggregation reproduces the published FORUM estimate.
check = policy_summary.merge(
    ranking_metrics,
    on=['policy_id', 'ordering', 'reply_mode', 'pinned', 'deployable', 'outcome', 'depth'],
    suffixes=('_published', '_recomputed'),
)
check['absolute_difference'] = (check['estimate'] - check['forum']).abs()
display(check['absolute_difference'].describe().to_frame('published_vs_recomputed_abs_difference'))
display(ranking_metrics.head())
print(f'{len(ranking_metrics):,} ranking-level rows; {ranking_metrics["ordering"].nunique()} orderings; {ranking_metrics["outcome"].nunique()} features.')

## Correlation and outlier helpers

Pearson correlation captures linear agreement; Spearman correlation captures agreement in ranking order. The outlier tables use the absolute vertical distance from the reference line or, for FORUM–nDCG, a robust residual from a within-panel linear fit. Labels are intended as a short-list for inspection, not as an automatic substantive classification.

In [ ]:
def correlation_row(frame, x, y, group_name='overall'):
    d = frame[[x, y]].dropna()
    if len(d) < 3 or d[x].nunique() < 2 or d[y].nunique() < 2:
        return {'group': group_name, 'n': len(d), 'pearson_r': np.nan, 'spearman_rho': np.nan}
    return {
        'group': group_name,
        'n': len(d),
        'pearson_r': d[x].corr(d[y], method='pearson'),
        'spearman_rho': d[x].corr(d[y], method='spearman'),
    }

def correlations_by(frame, x, y, group_cols):
    group_by = group_cols[0] if len(group_cols) == 1 else group_cols
    rows = [correlation_row(frame.loc[idx], x, y, ' | '.join(map(str, key if isinstance(key, tuple) else (key,))))
             for key, idx in frame.groupby(group_by, dropna=False).groups.items()]
    return pd.DataFrame(rows).sort_values('group').reset_index(drop=True)

def add_identity_line(ax, x, y):
    values = pd.concat([x, y]).dropna()
    if len(values):
        lo, hi = values.min(), values.max()
        ax.plot([lo, hi], [lo, hi], color='black', linestyle='--', linewidth=1, alpha=0.7)



## 2. Top-10 versus full-list FORUM

Each point is a ranking condition and feature. Colour denotes the feature, marker shape denotes reply handling, and filled versus hollow markers denote pinning status. Large departures from the dashed identity line are the first FORUM-behaviour outliers to inspect.

In [ ]:
forum_wide = (
    ranking_metrics
    .pivot_table(index=['policy_id', 'ordering', 'reply_mode', 'pinned', 'deployable', 'outcome', 'feature_label'],
                 columns='depth', values='forum', aggfunc='first')
    .reset_index()
)
forum_wide = forum_wide.rename(columns={'full': 'forum_full', 'top10': 'forum_top10'})
forum_wide = forum_wide.dropna(subset=['forum_full', 'forum_top10']).copy()
forum_wide['difference_top10_minus_full'] = forum_wide['forum_top10'] - forum_wide['forum_full']
forum_wide['abs_difference'] = forum_wide['difference_top10_minus_full'].abs()

display(correlation_row(forum_wide, 'forum_full', 'forum_top10'))
display(correlations_by(forum_wide, 'forum_full', 'forum_top10', ['feature_label']))

from matplotlib.lines import Line2D
fig, ax = plt.subplots(figsize=(15, 10))
feature_order = [feature for feature in FEATURE_LABELS.values() if feature in forum_wide['feature_label'].unique()]
feature_palette = dict(zip(feature_order, sns.color_palette('tab10', len(feature_order))))
reply_markers = {'loose': 'o', 'trees': 's', 'hidden': '^'}
for feature in feature_order:
    for reply_mode, marker in reply_markers.items():
        for pinned in (False, True):
            panel = forum_wide[
                forum_wide['feature_label'].eq(feature)
                & forum_wide['reply_mode'].eq(reply_mode)
                & forum_wide['pinned'].eq(pinned)
            ]
            if panel.empty:
                continue
            colour = feature_palette[feature]
            ax.scatter(panel['forum_full'], panel['forum_top10'], marker=marker, s=58, alpha=0.62,
                       facecolors=colour if pinned else 'none', edgecolors=colour, linewidths=0.9)
add_identity_line(ax, forum_wide['forum_full'], forum_wide['forum_top10'])
ax.set_xlabel('Full-list FORUM')
ax.set_ylabel('Top-10 FORUM')
ax.set_title('Top-10 versus full-list FORUM across ranking conditions')

feature_handles = [Line2D([0], [0], marker='o', linestyle='', color=colour,
                          markerfacecolor=colour, label=feature)
                   for feature, colour in feature_palette.items()]
reply_handles = [Line2D([0], [0], marker=marker, linestyle='', color='black', label=reply)
                 for reply, marker in reply_markers.items()]
pin_handles = [
    Line2D([0], [0], marker='o', linestyle='', color='black', markerfacecolor='none', label='Unpinned'),
    Line2D([0], [0], marker='o', linestyle='', color='black', markerfacecolor='black', label='Pinned'),
]
fig.legend(handles=feature_handles, title='Feature', loc='upper left',
           bbox_to_anchor=(0.80, 0.98), borderaxespad=0.0)
fig.legend(handles=reply_handles, title='Reply mode', loc='upper left',
           bbox_to_anchor=(0.80, 0.57), borderaxespad=0.0)
fig.legend(handles=pin_handles, title='Pin status', loc='upper left',
           bbox_to_anchor=(0.80, 0.34), borderaxespad=0.0)
fig.tight_layout(rect=(0, 0, 0.78, 1))
plt.show()

print('Largest top-10/full departures:')
display(forum_wide.nlargest(20, 'abs_difference')[['feature_label', 'ordering', 'reply_mode', 'pinned', 'forum_top10', 'forum_full', 'difference_top10_minus_full']])

## 3. FORUM versus nDCG

This uses the same ranking-level units as the previous section. The two depth panels make it possible to see whether apparent agreement is specific to the displayed top 10 or persists over the full visible list.

In [ ]:
forum_ndcg = ranking_metrics.copy()
display(correlation_row(forum_ndcg, 'forum', 'ndcg'))
display(correlations_by(forum_ndcg, 'forum', 'ndcg', ['depth', 'feature_label']))

from matplotlib.lines import Line2D
feature_order = [feature for feature in FEATURE_LABELS.values() if feature in forum_ndcg['feature_label'].unique()]
depth_order = ['top10', 'full']
reply_palette = {'loose': '#1f77b4', 'trees': '#ff7f0e', 'hidden': '#2ca02c'}
fig, axes = plt.subplots(len(feature_order), 2, figsize=(16, 4 * len(feature_order)), squeeze=False)
for row, feature in enumerate(feature_order):
    for col, depth in enumerate(depth_order):
        ax = axes[row, col]
        panel = forum_ndcg[(forum_ndcg['feature_label'].eq(feature)) & (forum_ndcg['depth'].eq(depth))].copy()
        sns.scatterplot(data=panel, x='forum', y='ndcg', hue='reply_mode', style='pinned',
                        palette=reply_palette, alpha=0.62, s=58, legend=False, ax=ax)
        if len(panel) >= 3 and panel['forum'].nunique() > 1:
            sns.regplot(data=panel, x='forum', y='ndcg', scatter=False, color='black',
                        line_kws={'linestyle': '--', 'linewidth': 1}, ax=ax)
        ax.set_title(f'{feature} — {depth}')
        ax.set_xlabel('Mean FORUM')
        ax.set_ylabel('Mean nDCG')

reply_handles = [Line2D([0], [0], marker='o', linestyle='', color=color, label=label)
                 for label, color in reply_palette.items()]
pin_handles = [
    Line2D([0], [0], marker='o', linestyle='', color='black', markerfacecolor='white', label='Unpinned'),
    Line2D([0], [0], marker='X', linestyle='', color='black', label='Pinned'),
]
fig.legend(reply_handles + pin_handles, [h.get_label() for h in reply_handles + pin_handles],
           loc='center left', bbox_to_anchor=(0.99, 0.5), title='Reply mode / pinning')
fig.suptitle('FORUM versus nDCG by feature and ranking depth', y=1.002)
fig.tight_layout(rect=(0, 0, 0.94, 0.995))
plt.show()

print('Largest within-depth FORUM/nDCG residuals:')
outlier_parts = []
for depth, panel in forum_ndcg.groupby('depth'):
    panel = panel.copy()
    slope, intercept = np.polyfit(panel['forum'], panel['ndcg'], 1)
    panel['abs_residual'] = (panel['ndcg'] - (intercept + slope * panel['forum'])).abs()
    outlier_parts.append(panel)
display(pd.concat(outlier_parts).nlargest(20, 'abs_residual')[['depth', 'feature_label', 'ordering', 'reply_mode', 'pinned', 'forum', 'ndcg', 'abs_residual']])

## 4. Regression FORUM on feature versus coefficient

Here `x` is the story-averaged FORUM of a regression audience/editor ranking on the feature outcome, and `y` is the corresponding regression coefficient. The canonical interface is loose, unpinned; top-10 and full-list values are shown separately.

The selected FORUM/ranking analysis run no longer carries all 41 regression features. This diagnostic therefore plots only selected outcomes with matching coefficient rows and explicitly reports the remaining selected outcomes as unmatched.

In [ ]:
regression_coefficients = pd.read_csv(PAPER1_TABLES / 'regression_selector_associations.csv')
regression_coefficients = regression_coefficients[regression_coefficients['scope'].eq('all')].copy()

selector_map = {
    'audience': ('audience_log_odds', 'regression_audience'),
    'editor': ('curator_log_odds', 'regression_editor'),
}
coefficient_long = pd.concat([
    regression_coefficients[['term', 'feature', 'audience_log_odds']].rename(columns={'term': 'outcome', 'audience_log_odds': 'coefficient'}).assign(selector='audience', ordering='regression_audience'),
    regression_coefficients[['term', 'feature', 'curator_log_odds']].rename(columns={'term': 'outcome', 'curator_log_odds': 'coefficient'}).assign(selector='editor', ordering='regression_editor'),
], ignore_index=True)

regression_forum = ranking_metrics.query(
    "ordering in ['regression_audience', 'regression_editor'] and reply_mode == 'loose' and not pinned"
)[['ordering', 'outcome', 'depth', 'forum', 'feature_label']].copy()
regression_plot = regression_forum.merge(coefficient_long, on=['ordering', 'outcome'], how='inner')
regression_plot['feature_label'] = regression_plot['feature'].where(regression_plot['feature'].notna(), regression_plot['feature_label'])

def regression_feature_category(feature):
    feature = str(feature)
    lower = feature.lower()
    if feature.startswith('AQuA '):
        return 'AQuA'
    if any(term in lower for term in ('length', 'lexical diversity', 'reading difficulty')):
        return 'Language / readability'
    if any(term in lower for term in ('sentiment', 'toxicity')):
        return 'Sentiment / toxicity'
    if any(term in lower for term in ('similarity', 'novelty')):
        return 'Semantic / novelty'
    if any(term in lower for term in ('url', 'reply', 'root')):
        return 'Comment structure'
    if 'author' in lower or 'prior author' in lower:
        return 'Author history'
    return 'Timing / discussion context'

regression_plot['feature_category'] = regression_plot['feature'].map(regression_feature_category)
regression_plot['odds_ratio'] = np.exp(regression_plot['coefficient'])

matched = sorted(regression_plot['outcome'].unique())
outcomes_without_coefficients = sorted(set(FEATURE_LABELS) - set(matched))
print(f'Matched outcome features: {len(matched)}')
print('Unmatched policy outcomes:', outcomes_without_coefficients)
display(regression_plot[['outcome', 'feature_label', 'selector', 'depth', 'forum', 'coefficient']].sort_values(['depth', 'selector', 'outcome']))

display(correlations_by(regression_plot, 'forum', 'coefficient', ['depth', 'selector']))
odds_ratio_correlations = correlations_by(regression_plot, 'forum', 'odds_ratio', ['depth', 'selector'])
print('Odds-ratio correlations (including Spearman rank correlation):')
display(odds_ratio_correlations)
fig, axes = plt.subplots(1, 2, figsize=(18, 8), sharey=True)
selector_order = ['audience', 'editor']
depth_palette = {'top10': '#1f77b4', 'full': '#d62728'}
for ax, selector in zip(axes, selector_order):
    panel = regression_plot[regression_plot['selector'].eq(selector)]
    sns.scatterplot(data=panel, x='forum', y='odds_ratio', hue='depth', style='feature_category',
                    palette=depth_palette, s=110, ax=ax)
    ax.axhline(1, color='black', linestyle=':', linewidth=1)
    ax.set_title(selector.title())
    ax.set_xlabel('Regression FORUM on feature')
    ax.set_ylabel('Exponentiated regression coefficient (odds ratio)')
    ax.legend_.remove()
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='center left', bbox_to_anchor=(0.98, 0.5), title='Depth / feature category')
fig.suptitle('Regression FORUM on feature versus coefficient', y=1.02)
fig.tight_layout()
plt.show()

## 5. ML FORUM versus SHAP — deliberately deferred

The current selected ML rankers have paired FORUM-ready policy scores, but this notebook does not calculate SHAP. The existing XGBoost TreeSHAP files are not automatically used because the comparison must be aligned to the exact selected model, selector, feature set, split, and prediction sample. There are no matching neural SHAP products in the model-ranker directories.

Before making the four requested ML figures (XGB metadata, XGB metadata + text, Neural metadata, Neural metadata + text), decide:

- whether SHAP is calculated on the sealed FORUM/ranking analysis test predictions or on development-fold out-of-fold predictions;
- how to choose a background set without leaking held-out information;
- whether to aggregate signed SHAP, mean absolute SHAP, or the audience-minus-editor SHAP contrast;
- whether comments are weighted equally or first averaged within story, matching the FORUM aggregation; and
- how to make TreeSHAP and neural-network SHAP comparable across feature sets and selectors.

The recommended next step is to generate SHAP from the exact selected ranker artifacts, on the same held-out comment rows used for the ranking metrics, using a training-only background, then aggregate to feature × selector × model-family × feature-set before joining to the ranking-level FORUM table.

In [ ]:
ml_orderings = [
    'xgb_metadata_audience', 'xgb_metadata_editor',
    'xgb_metadata_text_audience', 'xgb_metadata_text_editor',
    'neural_metadata_audience', 'neural_metadata_editor',
    'neural_metadata_text_audience', 'neural_metadata_text_editor',
]
ml_forum_plan = ranking_metrics.query(
    "ordering in @ml_orderings and reply_mode == 'loose' and not pinned"
)[['ordering', 'outcome', 'depth', 'forum', 'feature_label']].copy()
ml_forum_plan['model_family'] = np.where(ml_forum_plan['ordering'].str.startswith('xgb'), 'XGB', 'Neural')
ml_forum_plan['feature_set'] = np.where(ml_forum_plan['ordering'].str.contains('text'), 'metadata + text', 'metadata')
ml_forum_plan['selector'] = np.where(ml_forum_plan['ordering'].str.contains('audience'), 'audience', 'editor')

shap_candidates = [
    REPO_ROOT / 'model_output/selection_2025/xgboost_paper2/all/test_treeshap_sample.parquet',
    REPO_ROOT / 'model_output/selection_2025/xgboost_paper2/root/test_treeshap_sample.parquet',
    REPO_ROOT / 'model_output/selection_2025/xgboost/all/oof_treeshap_sample.parquet',
]
shap_inventory = pd.DataFrame({
    'candidate': [str(path.relative_to(REPO_ROOT)) for path in shap_candidates],
    'exists': [path.exists() for path in shap_candidates],
    'used_in_this_notebook': False,
})
display(ml_forum_plan.head(16))
display(shap_inventory)
display(Markdown('**No SHAP values were calculated or joined in this notebook.** The table above is only a provenance inventory and the ML FORUM table is a join plan for the follow-up analysis.'))

## Interpretation checklist

When reviewing the plots, prioritise points that are far from the relevant reference pattern, but keep the aggregation and design in view. A high correlation does not imply that FORUM and nDCG are interchangeable; the scientifically interesting cases are ranking conditions where one metric is high and the other is unexpectedly low, or where the top-10/full-list relationship changes sharply by feature.